# Quadrature Rules and Their Convergence

Every rule approximates $\int_a^b f \approx \sum_i w_i f(x_i)$. They differ in where the nodes go and how fast the error falls as $n$ grows.

In [ ]:
Pollis.packages("FastGaussQuadrature", "QuadGK", "Plots")

In [ ]:
using FastGaussQuadrature, QuadGK, Plots
f(x) = x^2 * exp(-x)
a, b = 0.0, 4.0
ref = quadgk(f, a, b; rtol=1e-14)[1]

In [ ]:
trapezoid(f, a, b, n) = (h = (b - a) / n; h * (sum(f, range(a, b, length=n + 1)) - (f(a) + f(b)) / 2))
midpoint(f, a, b, n) = (h = (b - a) / n; h * sum(i -> f(a + (i - 0.5) * h), 1:n))
function simpson(f, a, b, n)
    n += isodd(n); h = (b - a) / n; xs = range(a, b, length=n + 1)
    return h / 3 * (f(a) + f(b) + 4sum(f, xs[2:2:end-1]) + 2sum(f, xs[3:2:end-2]; init=0.0))
end
function gauss(f, a, b, n)
    t, w = gausslegendre(n)   # nodes and weights on [-1, 1]
    return (b - a) / 2 * sum(w .* f.((b - a) / 2 .* t .+ (a + b) / 2))
end

In [ ]:
ns = 2 .^ (1:10)
rules = ["trapezoid" => trapezoid, "midpoint" => midpoint, "simpson" => simpson, "gauss" => gauss]
p = plot(xscale=:log2, yscale=:log10, xlabel="n", ylabel="absolute error", legend=:bottomleft)
for (name, rule) in rules
    plot!(ns, [max(abs(rule(f, a, b, n) - ref), 1e-17) for n in ns], m=:circle, lw=2, label=name)
end
p

Trapezoid and midpoint fall with slope $-2$, Simpson with slope $-4$, and Gauss-Legendre reaches machine precision with about 15 nodes. Gauss with $n$ nodes is exact for polynomials of degree up to $2n - 1$:

In [ ]:
[(n = n, error_on_x_to_the_2n_minus_1 = gauss(x -> x^(2n - 1), 0, 1, n) - 1 / 2n, error_on_x_to_the_2n = gauss(x -> x^(2n), 0, 1, n) - 1 / (2n + 1)) for n in 1:5]

## Exercises

1. Repeat with $f(x) = \sqrt{x}$ on $[0, 1]$. Why do all rates drop?
2. Repeat with $f(x) = |x - 1.3|$ and put $1.3$ at a node.